In [1]:
# This is necessary to recognize the modules
import os
import sys
from decimal import Decimal
import warnings

warnings.filterwarnings("ignore")

root_path = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.append(root_path)

In [2]:
from core.backtesting import BacktestingEngine

backtesting = BacktestingEngine(root_path=root_path, load_cached_data=False)

In [3]:
from hummingbot.strategy_v2.executors.position_executor.data_types import TrailingStop
from controllers.directional_trading.elite_oro_1 import EliteOroConfig
import datetime
from decimal import Decimal


# Controller configuration
connector_name = "binance_perpetual"
trading_pair = "ADA-USDT"
interval = "1m"
total_amount_quote = 1000
max_executors_per_side = 2
take_profit = 0.31
stop_loss = 0.065
trailing_stop_activation_price = 0.029
trailing_stop_trailing_delta = 0.017
time_limit = 60 * 60 * 2  # 2 hours
cooldown_time = 60  # 1 minute

# Elite Oro specific parameters
zlema_fast = 7  # Range: 3-8
zlema_medium = 17  # Range: 8-21
zlema_slow = 19  # Range: 13-34
atr_length = 17  # Range: 10-21
atr_multiplier = 1.2000000000000002  # Range: 0.8-1.5
volume_ma_period = 25
volume_surge = 2.7  # Range: 1.5-3.0
price_channel_period = 11
rsi_period = 8

# Creating the instance of the configuration and the controller
config = EliteOroConfig(
    id=f"elite_oro_{connector_name}_{interval}_{trading_pair}",
    connector_name=connector_name,
    trading_pair=trading_pair,
    interval=interval,
    zlema_fast=zlema_fast,
    zlema_medium=zlema_medium,
    zlema_slow=zlema_slow,
    atr_length=atr_length,
    atr_multiplier=Decimal(atr_multiplier),
    volume_ma_period=volume_ma_period,
    volume_surge=Decimal(volume_surge),
    price_channel_period=price_channel_period,
    rsi_period=rsi_period,
    total_amount_quote=Decimal(total_amount_quote),
    take_profit=Decimal(take_profit),
    stop_loss=Decimal(stop_loss),
    trailing_stop=TrailingStop(
        activation_price=Decimal(trailing_stop_activation_price),
        trailing_delta=Decimal(trailing_stop_trailing_delta)
    ),
    time_limit=time_limit,
    max_executors_per_side=max_executors_per_side,
    cooldown_time=cooldown_time,
)

In [4]:
# Running the backtesting this will output a backtesting result object that has built in methods to visualize the results

start = int(datetime.datetime(2025, 1, 1).timestamp())
end = int(datetime.datetime(2025, 3, 1).timestamp())


backtesting_result = await backtesting.run_backtesting(config, start, end, "1m")

2025-04-26 01:06:57,987 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x179a4fe80>
2025-04-26 01:06:57,992 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x104125d20>, 132056.378202666)])']
connector: <aiohttp.connector.TCPConnector object at 0x179a4fe20>
2025-04-26 01:07:46,630 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x179d93e80>
2025-04-26 01:07:46,665 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x179d5ff40>, 132104.991677)])']
connector: <aiohttp.connector.TCPConnector object at 0x179d93eb0>
2025-04-26 01:08:22,580 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x179d93dc0>
2025-04-26 01:08:22,659 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHand

In [5]:
# Let's see what is inside the backtesting results
print(backtesting_result.get_results_summary())
backtesting_result.get_backtesting_figure()


Net PNL: $-122.74 (-12.27%) | Max Drawdown: $-148.18 (-14.58%)
Total Volume ($): 131000.00 | Sharpe Ratio: -0.27 | Profit Factor: 0.71
Total Executors: 131 | Accuracy Long: 0.42 | Accuracy Short: 0.42
Close Types: Take Profit: 0 | Stop Loss: 0 | Time Limit: 125 |
             Trailing Stop: 6 | Early Stop: 0



In [6]:
# 2. The executors dataframe: this is the dataframe that contains the information of the orders that were executed
import pandas as pd

executors_df = backtesting_result.executors_df
executors_df.head()

,id,timestamp,type,close_timestamp,close_type,status,config,net_pnl_pct,net_pnl_quote,cum_fees_quote,filled_amount_quote,is_active,is_trading,custom_info,controller_id,side
0,Cn6X1Yt7KRB77rwUryNcwiFpsnQLZsBcnSxGhKVLod4L,1735755480,position_executor,1735759500,CloseType.TRAILING_STOP,RunnableStatus.TERMINATED,{'id': 'Cn6X1Yt7KRB77rwUryNcwiFpsnQLZsBcnSxGhK...,0.03228288288288243268464583479726570658385753...,16.1414414414412163978340686298906803131103515625,0.29999999999999998889776975374843459576368331...,1000,False,False,"{'close_price': 0.9172, 'level_id': None, 'sid...",None,BUY
1,HAjfgSmvKv669WzvFoztpKT9EPoyYBJ19K6vdA9BWUkH,1735768500,position_executor,1735775700,CloseType.TIME_LIMIT,RunnableStatus.TERMINATED,{'id': 'HAjfgSmvKv669WzvFoztpKT9EPoyYBJ19K6vdA...,-0.0015776232891584023032560679666858050040900...,-0.7888116445792011655058217911573592573404312...,0.29999999999999998889776975374843459576368331...,1000,False,False,"{'close_price': 0.9197, 'level_id': None, 'sid...",None,BUY
2,DQjtm4zvEzLFHN5bD2gfwsXtYYpBSeaUXzZERsibe5x6,1735772460,position_executor,1735779660,CloseType.TIME_LIMIT,RunnableStatus.TERMINATED,{'id': 'DQjtm4zvEzLFHN5bD2gfwsXtYYpBSeaUXzZERs...,-0.0336578512396706469589346966131415683776140...,-16.828925619835320759420937974937260150909423...,0.29999999999999993338661852249060757458209991...,999.9999999999998863131622783839702606201171875,False,False,"{'close_price': 0.95, 'level_id': None, 'side'...",None,SELL
3,4sAZEsqEyGWord6zwaaojSaQqDyJGbQG5brfy3YGzqWG,1735792560,position_executor,1735799760,CloseType.TIME_LIMIT,RunnableStatus.TERMINATED,{'id': '4sAZEsqEyGWord6zwaaojSaQqDyJGbQG5brfy3...,-0.0130507821645205572547165928654067101888358...,-6.5253910822602785302137817780021578073501586...,0.29999999999999998889776975374843459576368331...,1000,False,False,"{'close_price': 0.928, 'level_id': None, 'side...",None,BUY
4,GZjK2t1sxquznTuE48AaknqoGs1UoPWMEZ9pEq7TV9Kq,1735846200,position_executor,1735853400,CloseType.TIME_LIMIT,RunnableStatus.TERMINATED,{'id': 'GZjK2t1sxquznTuE48AaknqoGs1UoPWMEZ9pEq...,-0.0029965822652913802481844385283693554811179...,-1.4982911326456902489923095345147885382175445...,0.29999999999999998889776975374843459576368331...,1000.0000000000001136868377216160297393798828125,False,False,"{'close_price': 0.962, 'level_id': None, 'side...",None,SELL


### Backtesting Analysis

### Scatter of PNL per Trade
This bar chart illustrates the PNL for each individual trade. Positive PNLs are shown in green and negative PNLs in red, providing a clear view of profitable vs. unprofitable trades.


In [7]:
import plotly.express as px

# Create a new column for profitability
executors_df['profitable'] = executors_df['net_pnl_quote'] > 0

# Create the scatter plot
fig = px.scatter(
    executors_df,
    x="timestamp",
    y='net_pnl_quote',
    title='PNL per Trade',
    color='profitable',
    color_discrete_map={True: 'green', False: 'red'},
    labels={'timestamp': 'Timestamp', 'net_pnl_quote': 'Net PNL (Quote)'},
    hover_data=['filled_amount_quote', 'side']
)

# Customize the layout
fig.update_layout(
    xaxis_title="Timestamp",
    yaxis_title="Net PNL (Quote)",
    legend_title="Profitable",
    font=dict(size=12, color="white"),
    showlegend=False,
    plot_bgcolor='rgba(0,0,0,0.8)',  # Dark background
    paper_bgcolor='rgba(0,0,0,0.8)',  # Dark background for the entire plot area
    xaxis=dict(gridcolor="gray"),
    yaxis=dict(gridcolor="gray")
)

# Add a horizontal line at y=0 to clearly separate profits and losses
fig.add_hline(y=0, line_dash="dash", line_color="lightgray")

# Show the plot
fig.show()

### Histogram of PNL Distribution
The histogram displays the distribution of PNL values across all trades. It helps in understanding the frequency and range of profit and loss outcomes.


In [8]:
fig = px.histogram(executors_df, x='net_pnl_quote', title='PNL Distribution')
fig.show()


# Conclusion
We can see that the indicator has potential to bring good signals to trade and might be interesting to see how we can design a market maker that shifts the mid price based on this indicator.
A lot of the short signals are wrong but if we zoom in into the loss signals we can see that the losses are not that big and the wins are bigger and if we had implemented the trailing stop feature probably a lot of them are going to be profits.

# Next steps
- Filter only the loss signals and understand what you can do to prevent them
- Try different configuration values for the indicator
- Test in multiple markets, pick mature markets like BTC-USDT or ETH-USDT and also volatile markets like DOGE-USDT or SHIB-USDT